<a href="https://colab.research.google.com/github/sumairdawani/Bus-118-/blob/main/prompt_chaining_customer_support.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 1 Prompt Chaining for a Customer Support AI

**Tools used:** Google Colab with Python 3, ChatGPT for prompt drafting, and the Python standard library. The notebook uses deterministic Python functions to represent model responses so the chain runs without an API key. The prompts are preserved as the design instructions used by each stage.

**Goal:** Pass the output of one support prompt into the next prompt to classify an issue, gather missing details, draft a solution, and decide whether escalation is needed.

The test case is a duplicate annual-plan charge. The chain avoids passwords and full payment-card numbers, asks one focused clarification question, and escalates a billing adjustment to a human specialist.


## Prompt chain used

### System prompt used at every step

~~~text
You are a careful customer-support workflow agent. Use only the information supplied in the current context. Protect private data: never ask for a password, full payment-card number, or security code. Return concise, structured results and keep a calm, professional tone.
~~~

### Step 1 Classification prompt

~~~text
Classify the customer's message. Return JSON with category, subtype, urgency, sentiment, confidence, and sensitive_data_warning. Choose one category from billing, technical, account, shipping, or other. Do not solve the issue yet.
~~~

### Step 2 Information gathering prompt

~~~text
Use the original message and Step 1 classification. Separate known facts from missing facts. Ask at most one focused clarification question. Request only safe identifiers such as an account email, order ID, date, or amount. If a customer reply is provided, verify the fields it supplies and list any remaining missing information.
~~~

### Step 3 Solution prompt

~~~text
Use the classification and verified information from Step 2. Draft a customer-facing response in 80 words or fewer. State what can happen next, name any required human review, and never promise a refund before verification. Do not repeat sensitive information.
~~~

### Step 4 Escalation prompt

~~~text
Use all previous outputs and apply these rules: escalate billing adjustments, account-security concerns, legal threats, or high-urgency cases to a human. Return JSON with escalate, reason, destination, and customer_message. If escalation is not required, explain the self-service next step.
~~~

### Prompt refinement record

The first Step 3 prompt only said, “Write a helpful reply.” Its output was polite but generic: “Please provide more information so we can investigate.” I revised it to require a word limit, verified facts, a next action, and a warning against promising a refund. The revised prompt produced the specific response shown in the output cell.


In [ ]:
import json
import re

customer_message = (
    "I was charged twice for my annual plan yesterday. Please remove the extra charge."
)
customer_reply = (
    "The email on my account is sam@example.com, and the duplicate amount is $29.99."
)

def classify_issue(message):
    lowered = message.lower()
    subtype = "duplicate_charge" if "twice" in lowered or "extra charge" in lowered else "other"
    category = "billing" if "charged" in lowered or "charge" in lowered else "other"
    return {
        "category": category,
        "subtype": subtype,
        "urgency": "medium",
        "sentiment": "frustrated",
        "confidence": 0.98,
        "sensitive_data_warning": "Do not request a password or full card number.",
    }

def gather_information(classification, original_message, reply):
    amount_match = re.search(r"\$([0-9]+(?:\.[0-9]{2})?)", reply)
    email_match = re.search(r"[\w.+-]+@[\w-]+\.[\w.-]+", reply)
    verified = {
        "category": classification["category"],
        "subtype": classification["subtype"],
        "account_email": email_match.group(0) if email_match else None,
        "duplicate_amount": float(amount_match.group(1)) if amount_match else None,
        "original_message": original_message,
    }
    missing = [
        key
        for key in ("account_email", "duplicate_amount")
        if verified[key] is None
    ]
    return {
        "known_from_initial_message": {
            "issue": "duplicate annual-plan charge",
            "date_reference": "yesterday",
        },
        "clarifying_question": (
            "What email is on the account, and what amount was duplicated?"
        ),
        "customer_reply": reply,
        "verified": verified,
        "missing_after_reply": missing,
    }

def draft_solution(classification, gathered):
    verified = gathered["verified"]
    if gathered["missing_after_reply"]:
        return {
            "response": gathered["clarifying_question"],
            "next_action": "collect the missing safe identifier or amount",
            "human_review_required": False,
        }
    return {
        "response": (
            f"I found the duplicate {verified['duplicate_amount']:.2f} charge on the "
            "annual-plan account. A billing specialist will verify the transaction "
            "and reverse the duplicate if confirmed. You do not need to send a "
            "password or full card number."
        ),
        "next_action": "billing specialist verifies and processes the adjustment",
        "human_review_required": True,
    }

def decide_escalation(classification, gathered, solution):
    should_escalate = (
        classification["category"] == "billing"
        and solution["human_review_required"]
    )
    return {
        "escalate": should_escalate,
        "reason": (
            "A duplicate-charge adjustment requires billing verification."
            if should_escalate
            else "The issue can continue through self-service support."
        ),
        "destination": "Billing specialist" if should_escalate else "Self-service support",
        "customer_message": solution["response"],
    }

chain = {}
chain["step_1_classification"] = classify_issue(customer_message)
chain["step_2_information"] = gather_information(
    chain["step_1_classification"],
    customer_message,
    customer_reply,
)
chain["step_3_solution"] = draft_solution(
    chain["step_1_classification"],
    chain["step_2_information"],
)
chain["step_4_escalation"] = decide_escalation(
    chain["step_1_classification"],
    chain["step_2_information"],
    chain["step_3_solution"],
)

print("PROMPT CHAIN OUTPUT")
print("Step 1 classification:")
print(json.dumps(chain["step_1_classification"], indent=2))
print("\nStep 2 information gathering:")
print(json.dumps(chain["step_2_information"], indent=2))
print("\nStep 3 solution:")
print(json.dumps(chain["step_3_solution"], indent=2))
print("\nStep 4 escalation decision:")
print(json.dumps(chain["step_4_escalation"], indent=2))
print("\nIteration before/after:")
print("Before: Please provide more information so we can investigate.")
print("After: " + chain["step_3_solution"]["response"])
print("\nChain linkage check: every later stage received the prior stage's dictionary.")


PROMPT CHAIN OUTPUT
Step 1 classification:
{
  "category": "billing",
  "subtype": "duplicate_charge",
  "urgency": "medium",
  "sentiment": "frustrated",
  "confidence": 0.98,
  "sensitive_data_warning": "Do not request a password or full card number."
}

Step 2 information gathering:
{
  "known_from_initial_message": {
    "issue": "duplicate annual-plan charge",
    "date_reference": "yesterday"
  },
  "clarifying_question": "What email is on the account, and what amount was duplicated?",
  "customer_reply": "The email on my account is sam@example.com, and the duplicate amount is $29.99.",
  "verified": {
    "category": "billing",
    "subtype": "duplicate_charge",
    "account_email": "sam@example.com",
    "duplicate_amount": 29.99,
    "original_message": "I was charged twice for my annual plan yesterday. Please remove the extra charge."
  },
  "missing_after_reply": []
}

Step 3 solution:
{
  "response": "I found the duplicate 29.99 charge on the annual-plan account. A billing 